# Treasury D+1 Forecasting — Construction of Predictable Series

This notebook builds a set of daily time series from transaction-level treasury inflows.

The goal is **not yet to forecast**. The goal is to create sensible, auditable series on which forecasting models can be trained:

- keep the most useful companies as individual series;
- split low-value / high-value movements because they likely have different regimes;
- aggregate sparse companies into behavior-based groups;
- keep `NO_COMPANY` separately;
- return daily series plus metadata so every series can be inspected.

## Core idea

The aggregate daily amount was not predictable enough because it mixes incompatible behaviors:

1. high-volume processors;
2. regular retail flows;
3. periodic/scheduled companies;
4. one-off or rare spikes;
5. unidentified counterparties.

So we build a **hybrid hierarchy**:

```text
Total amount
├── under_100k
│   ├── own company series
│   ├── behavior-based tail groups
│   └── NO_COMPANY
└── over_100k
    ├── own company series
    ├── behavior-based tail groups
    └── NO_COMPANY
```

The default threshold is `100_000`, but it is intentionally configurable and should be stress-tested.

## 0. Imports

In [ ]:
import re
import math
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 1. Configuration

Start with the default config, then tune it after you inspect the outputs.

Important defaults:

- `high_amount_threshold = 100_000`
- threshold is applied on `abs(amount)`, so it works whether your inflow sign convention is positive or negative
- companies with only 1 or 2 payments are not selected as own forecast series by default
- top companies are selected separately inside `under_100k` and `over_100k`
- remaining companies are grouped by behavior, frequency bin, value bin, and optionally movement-size bin

In [ ]:
@dataclass
class SeriesConfig:
    # Input columns
    date_col: str = "value_date"
    company_col: str = "company_name"
    amount_col: str = "amount"
    company_found_col: str = "company_found"

    # Main split
    high_amount_threshold: float = 100_000.0
    threshold_on_abs_amount: bool = True

    # Calendar
    # Use "D" if weekends/holidays are informative zeros.
    # Use "B" only if your treasury process never expects weekend movements.
    daily_freq: str = "D"

    # Own-company selection budget
    own_max_under_100k: int = 60
    own_max_over_100k: int = 40

    # Own-company eligibility
    own_min_payments_under_100k: int = 20
    own_min_active_days_under_100k: int = 10
    own_min_payments_over_100k: int = 3
    own_min_active_days_over_100k: int = 2

    # A rare company with 1 or 2 spikes is usually not forecastable alone.
    # It can still be monitored via tail groups and review tables.
    allow_one_or_two_spike_own_series: bool = False

    # Force/boost own selection by value, but only if the company is not a pure one/two-spike case.
    own_min_total_abs_under_100k: float = 500_000.0
    own_min_total_abs_over_100k: float = 500_000.0

    # Tail grouping
    tail_use_row_amount_bins: bool = True
    min_tail_series_rows: int = 3
    min_tail_series_total_abs: float = 25_000.0
    collapse_tiny_tail_groups: bool = True

    # Top samples shown in metadata
    max_companies_in_tail_sample: int = 8

    # Optional outputs
    output_dir: str = "series_outputs"

    # Row amount bins. These are separate from the main under/over split.
    under_row_amount_bins: tuple = (0, 1_000, 5_000, 20_000, 100_000)
    over_row_amount_bins: tuple = (100_000, 500_000, 1_000_000, 5_000_000, np.inf)


cfg = SeriesConfig()
asdict(cfg)

## 2. Load your dataframe

The notebook expects a dataframe named `df` with at least:

- `company_name`
- `value_date`
- `amount`
- `company_found`

Use either the CSV/Parquet loader below, or comment it out if `df` already exists in memory.

In [ ]:
# ---------------------------------------------------------------------
# Option A — load from file
# ---------------------------------------------------------------------
# DATA_PATH = "your_file.parquet"
# df = pd.read_parquet(DATA_PATH)

# DATA_PATH = "your_file.csv"
# df = pd.read_csv(DATA_PATH)

# ---------------------------------------------------------------------
# Option B — dataframe already exists
# ---------------------------------------------------------------------
# df = your_dataframe.copy()

# Safety check. Remove this assert after setting df.
assert "df" in globals(), "Please load your dataframe into a variable named df before continuing."

print(df.shape)
display(df.head())

## 3. Helpers

In [ ]:
def normalize_company_name(x) -> str:
    """Clean company names while keeping enough information for audit."""
    if pd.isna(x):
        return ""
    s = str(x).strip().upper()
    s = re.sub(r"\s+", " ", s)
    return s


def sanitize_label(x: str, max_len: int = 80) -> str:
    """Create a stable, readable label for column / series names."""
    s = normalize_company_name(x)
    s = re.sub(r"[^A-Z0-9]+", "_", s).strip("_")
    if not s:
        s = "UNKNOWN"
    return s[:max_len]


def normalized_entropy(values) -> float:
    """Entropy normalized to [0, 1]. Low means concentrated in one category."""
    counts = pd.Series(values).value_counts(dropna=True)
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum() / np.log(len(counts)))


def top_share(values) -> float:
    """Share of the most common value."""
    counts = pd.Series(values).value_counts(dropna=True)
    if counts.sum() == 0:
        return np.nan
    return float(counts.iloc[0] / counts.sum())


def safe_cv2(x) -> float:
    """Squared coefficient of variation."""
    arr = pd.Series(x).dropna().astype(float)
    if len(arr) <= 1:
        return np.nan
    mu = arr.mean()
    if abs(mu) < 1e-12:
        return np.nan
    return float((arr.std(ddof=1) / mu) ** 2)


def gap_stats(dates) -> dict:
    dates = pd.Series(pd.to_datetime(dates).dropna().unique()).sort_values()
    if len(dates) <= 1:
        return {
            "median_gap_days": np.nan,
            "mean_gap_days": np.nan,
            "cv_gap_days": np.nan,
            "min_gap_days": np.nan,
            "max_gap_days": np.nan,
        }
    gaps = dates.diff().dropna().dt.days.astype(float)
    mean_gap = gaps.mean()
    cv_gap = gaps.std(ddof=1) / mean_gap if mean_gap and not np.isnan(mean_gap) else np.nan
    return {
        "median_gap_days": float(gaps.median()),
        "mean_gap_days": float(mean_gap),
        "cv_gap_days": float(cv_gap) if not np.isnan(cv_gap) else np.nan,
        "min_gap_days": float(gaps.min()),
        "max_gap_days": float(gaps.max()),
    }


def bin_total_value(total_abs_amount: float) -> str:
    """Company-level value bin, based on total absolute amount in the split."""
    x = float(total_abs_amount)
    if x < 1_000:
        return "total_lt_1k"
    if x < 100_000:
        return "total_1k_100k"
    if x < 5_000_000:
        return "total_100k_5M"
    if x < 100_000_000:
        return "total_5M_100M"
    return "total_100M_plus"


def bin_frequency(n_payments: int) -> str:
    n = int(n_payments)
    if n <= 2:
        return "freq_1_2"
    if n <= 5:
        return "freq_3_5"
    if n <= 20:
        return "freq_6_20"
    if n <= 50:
        return "freq_21_50"
    if n <= 500:
        return "freq_51_500"
    return "freq_500_plus"


def sbc_label(adi: float, cv2: float) -> str:
    """
    Syntetos-Boylan-Croston-style demand pattern label.

    Use as a diagnostic feature, not as a hard truth:
    daily treasury company series are often zero-heavy, so most companies may look intermittent/lumpy.
    """
    if pd.isna(adi) or pd.isna(cv2):
        return "unknown"
    if adi < 1.32 and cv2 < 0.49:
        return "smooth"
    if adi < 1.32 and cv2 >= 0.49:
        return "erratic"
    if adi >= 1.32 and cv2 < 0.49:
        return "intermittent"
    return "lumpy"


def classify_behavior(row) -> str:
    """
    Business-oriented behavior class for grouping sparse tail companies.
    Tuned for treasury D+1 series construction, not inventory textbook classification.
    """
    if row["company_clean"] == "NO_COMPANY":
        return "no_company"

    n = row["n_payments"]
    active = row["n_active_days"]
    total_abs = row["total_abs_amount"]
    cv2 = row["cv2_pos_daily_abs_amount"]
    median_gap = row["median_gap_days"]
    cv_gap = row["cv_gap_days"]
    dow_top = row["dow_top_share"]
    dom_top = row["dom_top_share"]
    spike_ratio = row["max_to_median_payment_abs"]

    # Pure rare events
    if n <= 2:
        return "one_or_two_spikes"

    if n <= 5 and total_abs >= 100_000:
        return "rare_large"

    # Regular schedules
    monthly_regular = (
        active >= 4
        and (
            (not pd.isna(median_gap) and 25 <= median_gap <= 35 and (pd.isna(cv_gap) or cv_gap <= 0.35))
            or (not pd.isna(dom_top) and dom_top >= 0.60)
        )
    )
    if monthly_regular:
        return "periodic_monthly"

    weekly_regular = (
        active >= 8
        and (
            (not pd.isna(median_gap) and 6 <= median_gap <= 8 and (pd.isna(cv_gap) or cv_gap <= 0.45))
            or (not pd.isna(dow_top) and dow_top >= 0.60)
        )
    )
    if weekly_regular:
        return "periodic_weekly"

    # Frequent enough for learning a direct behavior
    if active >= 30:
        if not pd.isna(cv2) and cv2 <= 1.0:
            return "frequent_stable"
        return "frequent_erratic"

    # Intermittent/lumpy tails
    if active >= 6:
        if not pd.isna(cv2) and cv2 <= 0.49:
            return "intermittent_stable_amount"
        if not pd.isna(spike_ratio) and spike_ratio >= 10:
            return "lumpy_spiky"
        return "lumpy_intermittent"

    return "sparse_tail"


def make_row_amount_bin(abs_amount: float, split: str, cfg: SeriesConfig) -> str:
    """Movement-level amount bin used mainly for tail groups."""
    x = float(abs_amount)
    if split == "under_100k":
        bins = list(cfg.under_row_amount_bins)
        labels = [f"row_{int(bins[i])}_{int(bins[i+1])}" for i in range(len(bins) - 1)]
    else:
        bins = list(cfg.over_row_amount_bins)
        labels = []
        for i in range(len(bins) - 1):
            lo, hi = bins[i], bins[i + 1]
            if np.isinf(hi):
                labels.append(f"row_{int(lo)}_plus")
            else:
                labels.append(f"row_{int(lo)}_{int(hi)}")

    for i in range(len(bins) - 1):
        lo, hi = bins[i], bins[i + 1]
        if (x >= lo) and (x < hi):
            return labels[i]
    return labels[-1]

## 4. Prepare and validate raw data

This cell creates:

- clean company names;
- `NO_COMPANY` bucket;
- `under_100k` / `over_100k` split;
- movement-size bins.

In [ ]:
def prepare_transactions(df: pd.DataFrame, cfg: SeriesConfig) -> pd.DataFrame:
    d = df.copy()

    required = [cfg.date_col, cfg.company_col, cfg.amount_col]
    missing = [c for c in required if c not in d.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    d[cfg.date_col] = pd.to_datetime(d[cfg.date_col]).dt.normalize()
    d[cfg.amount_col] = pd.to_numeric(d[cfg.amount_col], errors="coerce")
    d = d.dropna(subset=[cfg.date_col, cfg.amount_col]).copy()

    if cfg.company_found_col in d.columns:
        found = d[cfg.company_found_col].fillna(False).astype(bool)
    else:
        found = d[cfg.company_col].notna() & (d[cfg.company_col].astype(str).str.strip() != "")

    d["company_found_clean"] = found
    d["company_clean_raw"] = d[cfg.company_col].map(normalize_company_name)
    d["company_clean"] = np.where(
        d["company_found_clean"] & (d["company_clean_raw"] != ""),
        d["company_clean_raw"],
        "NO_COMPANY",
    )

    d["amount"] = d[cfg.amount_col].astype(float)
    d["abs_amount"] = d["amount"].abs()

    threshold_basis = d["abs_amount"] if cfg.threshold_on_abs_amount else d["amount"]
    d["amount_regime"] = np.where(threshold_basis >= cfg.high_amount_threshold, "over_100k", "under_100k")

    d["row_amount_bin"] = [
        make_row_amount_bin(a, s, cfg) for a, s in zip(d["abs_amount"].values, d["amount_regime"].values)
    ]

    # Convenience date features for later diagnostics
    d["dayofweek"] = d[cfg.date_col].dt.dayofweek
    d["dayofmonth"] = d[cfg.date_col].dt.day
    d["month"] = d[cfg.date_col].dt.month
    d["is_month_end_window"] = d[cfg.date_col].dt.is_month_end | (d[cfg.date_col].dt.day >= 28)
    d["is_month_start_window"] = d[cfg.date_col].dt.day <= 3

    return d


tx = prepare_transactions(df, cfg)

print("Rows:", len(tx))
print("Date range:", tx[cfg.date_col].min(), "->", tx[cfg.date_col].max())
print("Unique cleaned companies:", tx["company_clean"].nunique())
print("Company found rate:", tx["company_found_clean"].mean().round(3))
display(tx.head())

display(
    tx.groupby("amount_regime")
      .agg(
          rows=("amount", "size"),
          companies=("company_clean", "nunique"),
          total_amount=("amount", "sum"),
          total_abs_amount=("abs_amount", "sum"),
          mean_abs_amount=("abs_amount", "mean"),
          max_abs_amount=("abs_amount", "max"),
      )
      .sort_index()
)

## 5. Company-level feature table

We compute features per `(amount_regime, company)`.

These features drive:

- own-series candidate selection;
- tail grouping;
- diagnostic review.

Key features:

- payment count;
- active days;
- total absolute amount;
- ADI = average interval between non-zero active days;
- CV² of positive daily absolute amounts;
- gap regularity;
- day-of-week and day-of-month concentration;
- spike concentration.

In [ ]:
def build_company_features(tx: pd.DataFrame, cfg: SeriesConfig) -> pd.DataFrame:
    date_col = cfg.date_col
    start = tx[date_col].min()
    end = tx[date_col].max()
    calendar = pd.date_range(start, end, freq=cfg.daily_freq)
    n_calendar_days = len(calendar)

    # Raw payment-level aggregates
    g = tx.groupby(["amount_regime", "company_clean"], dropna=False)

    base = g.agg(
        n_payments=("amount", "size"),
        n_active_days=(date_col, "nunique"),
        total_amount=("amount", "sum"),
        total_abs_amount=("abs_amount", "sum"),
        mean_payment_abs=("abs_amount", "mean"),
        median_payment_abs=("abs_amount", "median"),
        p90_payment_abs=("abs_amount", lambda x: x.quantile(0.90)),
        p99_payment_abs=("abs_amount", lambda x: x.quantile(0.99)),
        max_payment_abs=("abs_amount", "max"),
        first_date=(date_col, "min"),
        last_date=(date_col, "max"),
        company_found_rate=("company_found_clean", "mean"),
        dow_entropy=("dayofweek", normalized_entropy),
        dow_top_share=("dayofweek", top_share),
        dom_entropy=("dayofmonth", normalized_entropy),
        dom_top_share=("dayofmonth", top_share),
        month_end_share=("is_month_end_window", "mean"),
        month_start_share=("is_month_start_window", "mean"),
    ).reset_index()

    # Concentration of abs payment amounts: close to 1 means one movement dominates.
    hhi = (
        tx.assign(abs_share_tmp=lambda x: x["abs_amount"])
          .groupby(["amount_regime", "company_clean"])["abs_share_tmp"]
          .apply(lambda s: float(((s / s.sum()) ** 2).sum()) if s.sum() else np.nan)
          .rename("payment_abs_hhi")
          .reset_index()
    )
    base = base.merge(hhi, on=["amount_regime", "company_clean"], how="left")

    # Daily positive amount features per company/split.
    daily = (
        tx.groupby(["amount_regime", "company_clean", date_col], dropna=False)
          .agg(daily_amount=("amount", "sum"), daily_abs_amount=("abs_amount", "sum"))
          .reset_index()
    )

    daily_pos = daily[daily["daily_abs_amount"] > 0].copy()

    daily_feat = (
        daily_pos.groupby(["amount_regime", "company_clean"], dropna=False)
        .agg(
            mean_pos_daily_abs_amount=("daily_abs_amount", "mean"),
            median_pos_daily_abs_amount=("daily_abs_amount", "median"),
            std_pos_daily_abs_amount=("daily_abs_amount", "std"),
            max_pos_daily_abs_amount=("daily_abs_amount", "max"),
            cv2_pos_daily_abs_amount=("daily_abs_amount", safe_cv2),
        )
        .reset_index()
    )
    base = base.merge(daily_feat, on=["amount_regime", "company_clean"], how="left")

    # Gap stats
    gaps = (
        daily_pos.groupby(["amount_regime", "company_clean"], dropna=False)[date_col]
        .apply(gap_stats)
        .reset_index()
    )
    gaps_expanded = pd.concat(
        [
            gaps[["amount_regime", "company_clean"]],
            pd.json_normalize(gaps[date_col]),
        ],
        axis=1,
    )
    base = base.merge(gaps_expanded, on=["amount_regime", "company_clean"], how="left")

    base["history_days"] = n_calendar_days
    base["adi"] = base["history_days"] / base["n_active_days"].replace(0, np.nan)
    base["sbc_label"] = [sbc_label(a, c) for a, c in zip(base["adi"], base["cv2_pos_daily_abs_amount"])]

    base["span_days"] = (base["last_date"] - base["first_date"]).dt.days + 1
    base["days_since_last"] = (end - base["last_date"]).dt.days
    base["max_to_median_payment_abs"] = base["max_payment_abs"] / base["median_payment_abs"].replace(0, np.nan)

    base["freq_bin"] = base["n_payments"].map(bin_frequency)
    base["value_bin"] = base["total_abs_amount"].map(bin_total_value)

    # Split shares
    total_abs_all = base["total_abs_amount"].sum()
    split_totals = base.groupby("amount_regime")["total_abs_amount"].transform("sum")

    base["share_total_abs"] = base["total_abs_amount"] / total_abs_all if total_abs_all else np.nan
    base["share_split_abs"] = base["total_abs_amount"] / split_totals.replace(0, np.nan)

    # Behavior label
    base["behavior_label"] = base.apply(classify_behavior, axis=1)

    return base.sort_values(["amount_regime", "total_abs_amount"], ascending=[True, False]).reset_index(drop=True)


company_features = build_company_features(tx, cfg)

display(company_features.head(30))

display(
    company_features.groupby(["amount_regime", "freq_bin", "behavior_label"], dropna=False)
    .agg(
        companies=("company_clean", "nunique"),
        total_abs_amount=("total_abs_amount", "sum"),
        n_payments=("n_payments", "sum"),
    )
    .reset_index()
    .sort_values(["amount_regime", "total_abs_amount"], ascending=[True, False])
    .head(80)
)

## 6. Visual diagnostic: company landscape

Use these plots to understand whether the chosen threshold and groupings make sense.

Expected patterns in your case:

- few companies dominate amount;
- many companies have few payments;
- high-value rare companies should generally be grouped unless they have repeated behavior.

In [ ]:
def plot_company_landscape(company_features: pd.DataFrame):
    f = company_features[company_features["company_clean"] != "NO_COMPANY"].copy()

    fig, ax = plt.subplots(figsize=(10, 6))
    for regime in sorted(f["amount_regime"].unique()):
        s = f[f["amount_regime"] == regime]
        ax.scatter(
            s["n_payments"],
            s["total_abs_amount"],
            alpha=0.55,
            label=regime,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of payments, log scale")
    ax.set_ylabel("Total absolute amount, log scale")
    ax.set_title("Company landscape: occurrence vs total amount")
    ax.legend()
    plt.show()

    display(
        f.groupby(["amount_regime", "freq_bin"])
         .agg(companies=("company_clean", "nunique"), total_abs_amount=("total_abs_amount", "sum"))
         .assign(avg_abs_per_company=lambda x: x["total_abs_amount"] / x["companies"])
         .reset_index()
         .sort_values(["amount_regime", "total_abs_amount"], ascending=[True, False])
    )


plot_company_landscape(company_features)

## 7. Select companies that deserve their own series

The default is a **hybrid occurrence/value score**, but with guardrails:

- `NO_COMPANY` is never selected as an own company;
- 1–2 spike companies are not selected by default;
- `under_100k` needs more observations to be own-series eligible;
- `over_100k` can be selected with fewer observations because each movement matters more;
- final count is capped separately for under/over regimes.

You can later sweep the caps and thresholds to see how many series you get.

In [ ]:
def add_own_company_selection(company_features: pd.DataFrame, cfg: SeriesConfig) -> pd.DataFrame:
    f = company_features.copy()

    # Ranking features within split. Higher is better.
    f["rank_value_pct"] = f.groupby("amount_regime")["total_abs_amount"].rank(pct=True)
    f["rank_payments_pct"] = f.groupby("amount_regime")["n_payments"].rank(pct=True)
    f["rank_active_days_pct"] = f.groupby("amount_regime")["n_active_days"].rank(pct=True)

    # Recency bonus: recently active companies are more relevant for D+1.
    max_days = max(float(f["days_since_last"].max()), 1.0)
    f["recency_score"] = 1.0 - (f["days_since_last"].clip(lower=0) / max_days)

    # Regularity bonus: regular companies may be forecastable even if not the highest amount.
    f["regularity_bonus"] = f["behavior_label"].isin(["periodic_monthly", "periodic_weekly", "frequent_stable"]).astype(float)

    f["own_score"] = (
        0.45 * f["rank_value_pct"].fillna(0)
        + 0.25 * f["rank_payments_pct"].fillna(0)
        + 0.15 * f["rank_active_days_pct"].fillna(0)
        + 0.10 * f["recency_score"].fillna(0)
        + 0.05 * f["regularity_bonus"].fillna(0)
    )

    f["own_eligible"] = False
    f["own_reason"] = ""

    is_named = f["company_clean"] != "NO_COMPANY"
    not_one_two = (f["n_payments"] > 2) | cfg.allow_one_or_two_spike_own_series

    under_ok = (
        (f["amount_regime"] == "under_100k")
        & is_named
        & not_one_two
        & (
            (
                (f["n_payments"] >= cfg.own_min_payments_under_100k)
                & (f["n_active_days"] >= cfg.own_min_active_days_under_100k)
            )
            | (
                (f["total_abs_amount"] >= cfg.own_min_total_abs_under_100k)
                & (f["n_active_days"] >= cfg.own_min_active_days_under_100k)
            )
        )
    )

    over_ok = (
        (f["amount_regime"] == "over_100k")
        & is_named
        & not_one_two
        & (
            (
                (f["n_payments"] >= cfg.own_min_payments_over_100k)
                & (f["n_active_days"] >= cfg.own_min_active_days_over_100k)
            )
            | (
                (f["total_abs_amount"] >= cfg.own_min_total_abs_over_100k)
                & (f["n_active_days"] >= cfg.own_min_active_days_over_100k)
            )
        )
    )

    f.loc[under_ok | over_ok, "own_eligible"] = True

    selected_idx = []

    for regime, max_n in [("under_100k", cfg.own_max_under_100k), ("over_100k", cfg.own_max_over_100k)]:
        cand = f[(f["amount_regime"] == regime) & (f["own_eligible"])].copy()
        cand = cand.sort_values(["own_score", "total_abs_amount", "n_payments"], ascending=False)
        selected_idx.extend(cand.head(max_n).index.tolist())

    f["is_own_series"] = False
    f.loc[selected_idx, "is_own_series"] = True

    f.loc[f["is_own_series"], "own_reason"] = "selected_by_hybrid_occurrence_value_score"
    f.loc[(f["own_eligible"]) & (~f["is_own_series"]), "own_reason"] = "eligible_but_outside_series_budget"

    # Explain common exclusion cases.
    f.loc[(f["company_clean"] == "NO_COMPANY"), "own_reason"] = "no_company_bucket"
    f.loc[
        (f["company_clean"] != "NO_COMPANY") & (f["n_payments"] <= 2) & (~cfg.allow_one_or_two_spike_own_series),
        "own_reason",
    ] = "one_or_two_spikes_grouped_not_own"

    return f.sort_values(["amount_regime", "is_own_series", "own_score"], ascending=[True, False, False])


company_features = add_own_company_selection(company_features, cfg)

display(
    company_features[company_features["is_own_series"]]
    .sort_values(["amount_regime", "own_score"], ascending=[True, False])
    .loc[
        :,
        [
            "amount_regime",
            "company_clean",
            "n_payments",
            "n_active_days",
            "total_amount",
            "total_abs_amount",
            "share_split_abs",
            "behavior_label",
            "freq_bin",
            "value_bin",
            "own_score",
            "own_reason",
        ],
    ]
)

print("Number of own company series by split:")
display(company_features.groupby(["amount_regime", "is_own_series"]).size().rename("companies").reset_index())

## 8. Review important companies that were NOT selected

This is one of the most important audit tables.

Look especially for:

- high total amount but only one/two spikes;
- companies just outside the own-series budget;
- companies with strong periodicity that were not selected;
- suspicious `NO_COMPANY` amount.

In [ ]:
review_not_selected = (
    company_features[~company_features["is_own_series"]]
    .sort_values(["total_abs_amount", "n_payments"], ascending=False)
    .loc[
        :,
        [
            "amount_regime",
            "company_clean",
            "n_payments",
            "n_active_days",
            "total_amount",
            "total_abs_amount",
            "share_split_abs",
            "behavior_label",
            "freq_bin",
            "value_bin",
            "adi",
            "cv2_pos_daily_abs_amount",
            "median_gap_days",
            "cv_gap_days",
            "dow_top_share",
            "dom_top_share",
            "own_score",
            "own_reason",
        ],
    ]
)

display(review_not_selected.head(80))

## 9. Assign every transaction to a forecast series

Assignment rules:

1. `NO_COMPANY` stays separate by regime and movement-size bin.
2. Selected companies get their own series inside the regime.
3. All other companies go to behavior-based tail groups.

For tail rows, the default series key is:

```text
{under/over}__TAIL__{behavior}__{freq_bin}__{value_bin}__{row_amount_bin}
```

Tiny tail groups can be collapsed to avoid hundreds of unusably small series.

In [ ]:
def assign_series_ids(tx: pd.DataFrame, company_features: pd.DataFrame, cfg: SeriesConfig) -> pd.DataFrame:
    key_cols = ["amount_regime", "company_clean"]
    feature_cols = [
        "is_own_series",
        "behavior_label",
        "freq_bin",
        "value_bin",
        "total_abs_amount",
        "n_payments",
        "n_active_days",
        "own_score",
        "own_reason",
    ]

    assigned = tx.merge(
        company_features[key_cols + feature_cols],
        on=key_cols,
        how="left",
        validate="many_to_one",
    )

    def initial_series(row):
        regime = row["amount_regime"]
        company = row["company_clean"]
        rowbin = row["row_amount_bin"]

        if company == "NO_COMPANY":
            if cfg.tail_use_row_amount_bins:
                return f"{regime}__NO_COMPANY__{rowbin}"
            return f"{regime}__NO_COMPANY"

        if bool(row["is_own_series"]):
            return f"{regime}__OWN__{sanitize_label(company)}"

        parts = [
            regime,
            "TAIL",
            str(row["behavior_label"]),
            str(row["freq_bin"]),
            str(row["value_bin"]),
        ]
        if cfg.tail_use_row_amount_bins:
            parts.append(str(rowbin))
        return "__".join(parts)

    assigned["initial_series_id"] = assigned.apply(initial_series, axis=1)

    assigned["series_id"] = assigned["initial_series_id"]

    if cfg.collapse_tiny_tail_groups:
        info = (
            assigned.groupby("initial_series_id")
            .agg(
                n_rows=("amount", "size"),
                total_abs_amount=("abs_amount", "sum"),
                amount_regime=("amount_regime", "first"),
            )
            .reset_index()
        )

        is_tail = info["initial_series_id"].str.contains("__TAIL__")
        tiny = is_tail & (
            (info["n_rows"] < cfg.min_tail_series_rows)
            | (info["total_abs_amount"] < cfg.min_tail_series_total_abs)
        )
        tiny_ids = set(info.loc[tiny, "initial_series_id"])

        tiny_mask = assigned["initial_series_id"].isin(tiny_ids)
        assigned.loc[tiny_mask, "series_id"] = (
            assigned.loc[tiny_mask, "amount_regime"] + "__TAIL__MISC_TINY"
        )

    # Series type
    assigned["series_type"] = np.select(
        [
            assigned["series_id"].str.contains("__OWN__"),
            assigned["series_id"].str.contains("__NO_COMPANY"),
            assigned["series_id"].str.contains("__TAIL__"),
        ],
        ["own_company", "no_company", "tail_group"],
        default="other",
    )

    return assigned


assigned = assign_series_ids(tx, company_features, cfg)

print("Number of created series:", assigned["series_id"].nunique())
display(assigned[["value_date", "company_clean", "amount", "abs_amount", "amount_regime", "row_amount_bin", "series_id"]].head(20))
display(assigned["series_type"].value_counts())

## 10. Build daily long and wide time series

Outputs:

- `series_long`: one row per `(date, series_id)`;
- `series_wide`: one column per series;
- `series_info`: metadata for each series;
- `company_to_series`: audit mapping from company/split to series.

In [ ]:
def build_series_outputs(assigned: pd.DataFrame, cfg: SeriesConfig):
    date_col = cfg.date_col

    calendar = pd.date_range(
        assigned[date_col].min(),
        assigned[date_col].max(),
        freq=cfg.daily_freq,
        name=date_col,
    )

    daily_sparse = (
        assigned.groupby([date_col, "series_id"], dropna=False)
        .agg(amount=("amount", "sum"), abs_amount=("abs_amount", "sum"), n_rows=("amount", "size"))
        .reset_index()
    )

    series_ids = sorted(assigned["series_id"].unique())

    full_index = pd.MultiIndex.from_product([calendar, series_ids], names=[date_col, "series_id"])

    series_long = (
        daily_sparse.set_index([date_col, "series_id"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    # Wide signed amount table for modeling
    series_wide = (
        series_long.pivot(index=date_col, columns="series_id", values="amount")
        .sort_index()
        .fillna(0.0)
    )

    # Series metadata
    def top_companies_sample(sdf):
        tmp = (
            sdf.groupby("company_clean")["abs_amount"]
            .sum()
            .sort_values(ascending=False)
        )
        names = tmp.head(cfg.max_companies_in_tail_sample).index.tolist()
        return " | ".join(names)

    def top_company_share(sdf):
        tmp = sdf.groupby("company_clean")["abs_amount"].sum()
        if tmp.sum() == 0:
            return np.nan
        return float(tmp.max() / tmp.sum())

    series_info = (
        assigned.groupby("series_id")
        .apply(
            lambda sdf: pd.Series(
                {
                    "amount_regime": sdf["amount_regime"].iloc[0],
                    "series_type": sdf["series_type"].iloc[0],
                    "n_rows": len(sdf),
                    "n_companies": sdf["company_clean"].nunique(),
                    "n_active_days": sdf[date_col].nunique(),
                    "total_amount": sdf["amount"].sum(),
                    "total_abs_amount": sdf["abs_amount"].sum(),
                    "mean_abs_movement": sdf["abs_amount"].mean(),
                    "max_abs_movement": sdf["abs_amount"].max(),
                    "first_date": sdf[date_col].min(),
                    "last_date": sdf[date_col].max(),
                    "top_companies_sample": top_companies_sample(sdf),
                    "top_company_abs_share": top_company_share(sdf),
                }
            )
        )
        .reset_index()
    )

    series_info["share_total_abs"] = series_info["total_abs_amount"] / series_info["total_abs_amount"].sum()
    series_info["share_regime_abs"] = (
        series_info["total_abs_amount"]
        / series_info.groupby("amount_regime")["total_abs_amount"].transform("sum")
    )
    series_info = series_info.sort_values("total_abs_amount", ascending=False).reset_index(drop=True)

    company_to_series = (
        assigned.groupby(["amount_regime", "company_clean", "series_id"], dropna=False)
        .agg(
            n_rows=("amount", "size"),
            n_active_days=(date_col, "nunique"),
            total_amount=("amount", "sum"),
            total_abs_amount=("abs_amount", "sum"),
            series_type=("series_type", "first"),
        )
        .reset_index()
        .sort_values(["total_abs_amount", "n_rows"], ascending=False)
    )

    # Reconstruction check: series must sum exactly to raw daily total.
    raw_daily = assigned.groupby(date_col)["amount"].sum().reindex(calendar, fill_value=0.0)
    recon_daily = series_wide.sum(axis=1).reindex(calendar, fill_value=0.0)
    max_abs_reconstruction_error = float((raw_daily - recon_daily).abs().max())

    diagnostics = {
        "n_series": int(len(series_ids)),
        "n_own_company_series": int((series_info["series_type"] == "own_company").sum()),
        "n_tail_group_series": int((series_info["series_type"] == "tail_group").sum()),
        "n_no_company_series": int((series_info["series_type"] == "no_company").sum()),
        "max_abs_reconstruction_error": max_abs_reconstruction_error,
        "date_min": str(calendar.min().date()),
        "date_max": str(calendar.max().date()),
    }

    return {
        "series_long": series_long,
        "series_wide": series_wide,
        "series_info": series_info,
        "company_to_series": company_to_series,
        "assigned_transactions": assigned,
        "diagnostics": diagnostics,
    }


outputs = build_series_outputs(assigned, cfg)

print(outputs["diagnostics"])
display(outputs["series_info"].head(50))
display(outputs["company_to_series"].head(80))

## 11. Coverage diagnostics

Check whether the created series are usable:

- how much amount is covered by own-company series;
- how much is in tail groups;
- whether `NO_COMPANY` is material;
- whether too many tiny tail series remain.

In [ ]:
series_info = outputs["series_info"]

coverage = (
    series_info.groupby(["amount_regime", "series_type"])
    .agg(
        n_series=("series_id", "nunique"),
        total_amount=("total_amount", "sum"),
        total_abs_amount=("total_abs_amount", "sum"),
        n_rows=("n_rows", "sum"),
        n_companies=("n_companies", "sum"),
    )
    .reset_index()
)
coverage["share_abs_all"] = coverage["total_abs_amount"] / coverage["total_abs_amount"].sum()
coverage["share_abs_regime"] = (
    coverage["total_abs_amount"]
    / coverage.groupby("amount_regime")["total_abs_amount"].transform("sum")
)

display(coverage.sort_values(["amount_regime", "total_abs_amount"], ascending=[True, False]))

fig, ax = plt.subplots(figsize=(10, 5))
tmp = coverage.copy()
tmp["label"] = tmp["amount_regime"] + " / " + tmp["series_type"]
ax.bar(tmp["label"], tmp["total_abs_amount"])
ax.set_title("Total absolute amount by constructed series type")
ax.set_ylabel("Total absolute amount")
ax.tick_params(axis="x", rotation=45)
plt.show()

display(
    series_info.sort_values("total_abs_amount", ascending=True)
    .head(30)
    [["series_id", "series_type", "amount_regime", "n_rows", "n_companies", "total_abs_amount", "top_companies_sample"]]
)

## 12. Visual inspection of constructed series

Use these functions to inspect individual or top series.

Look for:

- series that should be split further;
- tail groups dominated by one company;
- periodic patterns hidden inside a tail group;
- suspicious unidentified company amounts;
- high-value rare events that need external data rather than pure time-series forecasting.

In [ ]:
def plot_top_series(outputs, top_n=12, series_type=None, amount_regime=None):
    series_info = outputs["series_info"].copy()
    wide = outputs["series_wide"]

    f = series_info.copy()
    if series_type is not None:
        f = f[f["series_type"].eq(series_type)]
    if amount_regime is not None:
        f = f[f["amount_regime"].eq(amount_regime)]

    ids = f.sort_values("total_abs_amount", ascending=False)["series_id"].head(top_n).tolist()

    for sid in ids:
        fig, ax = plt.subplots(figsize=(14, 4))
        wide[sid].plot(ax=ax)
        meta = series_info[series_info["series_id"] == sid].iloc[0]
        ax.set_title(
            f"{sid}\n"
            f"type={meta['series_type']} | rows={meta['n_rows']} | "
            f"companies={meta['n_companies']} | total_abs={meta['total_abs_amount']:,.0f}"
        )
        ax.set_xlabel("Date")
        ax.set_ylabel("Amount")
        plt.show()


def plot_one_series(outputs, series_id):
    wide = outputs["series_wide"]
    series_info = outputs["series_info"]

    if series_id not in wide.columns:
        raise ValueError(f"Unknown series_id: {series_id}")

    meta = series_info[series_info["series_id"] == series_id].iloc[0]
    fig, ax = plt.subplots(figsize=(14, 4))
    wide[series_id].plot(ax=ax)
    ax.set_title(
        f"{series_id}\n"
        f"type={meta['series_type']} | rows={meta['n_rows']} | "
        f"companies={meta['n_companies']} | total_abs={meta['total_abs_amount']:,.0f}"
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Amount")
    plt.show()

    display(meta.to_frame("value"))


# Examples:
plot_top_series(outputs, top_n=8)
# plot_top_series(outputs, top_n=8, series_type="tail_group")
# plot_top_series(outputs, top_n=8, amount_regime="over_100k")
# plot_one_series(outputs, outputs["series_info"]["series_id"].iloc[0])

## 13. Tail-group audit: groups dominated by one company

A tail group dominated by one company may deserve to be split, especially if:

- total amount is high;
- the dominant company has repeated observations;
- the pattern is periodic or otherwise forecastable.

However, one/two-spike companies may still be better handled as a tail risk group, not individual forecasts.

In [ ]:
tail_audit = (
    outputs["series_info"]
    .query("series_type == 'tail_group'")
    .sort_values(["top_company_abs_share", "total_abs_amount"], ascending=False)
    [[
        "series_id",
        "amount_regime",
        "n_rows",
        "n_companies",
        "total_abs_amount",
        "top_company_abs_share",
        "top_companies_sample",
    ]]
)

display(tail_audit.head(80))

## 14. Optional: construct alternative granularities

Run compact/balanced/detailed scenarios and compare the number of series and coverage.

This lets you decide how many series you want to forecast before doing any modeling.

In [ ]:
def run_series_construction_scenario(df, cfg_updates: dict, scenario_name: str):
    scenario_cfg = SeriesConfig(**{**asdict(cfg), **cfg_updates})
    tx_s = prepare_transactions(df, scenario_cfg)
    cf_s = build_company_features(tx_s, scenario_cfg)
    cf_s = add_own_company_selection(cf_s, scenario_cfg)
    assigned_s = assign_series_ids(tx_s, cf_s, scenario_cfg)
    outputs_s = build_series_outputs(assigned_s, scenario_cfg)

    info = outputs_s["series_info"]
    summary = {
        "scenario": scenario_name,
        **outputs_s["diagnostics"],
        "own_abs_share": float(info.loc[info["series_type"].eq("own_company"), "total_abs_amount"].sum() / info["total_abs_amount"].sum()),
        "tail_abs_share": float(info.loc[info["series_type"].eq("tail_group"), "total_abs_amount"].sum() / info["total_abs_amount"].sum()),
        "no_company_abs_share": float(info.loc[info["series_type"].eq("no_company"), "total_abs_amount"].sum() / info["total_abs_amount"].sum()),
    }
    return summary, outputs_s


SCENARIOS = {
    "compact": {
        "own_max_under_100k": 25,
        "own_max_over_100k": 15,
        "min_tail_series_total_abs": 100_000,
    },
    "balanced": {
        "own_max_under_100k": 60,
        "own_max_over_100k": 40,
        "min_tail_series_total_abs": 25_000,
    },
    "detailed": {
        "own_max_under_100k": 120,
        "own_max_over_100k": 80,
        "min_tail_series_total_abs": 5_000,
    },
}

scenario_summaries = []
scenario_outputs = {}

for name, updates in SCENARIOS.items():
    summary, out = run_series_construction_scenario(df, updates, name)
    scenario_summaries.append(summary)
    scenario_outputs[name] = out

scenario_summary_df = pd.DataFrame(scenario_summaries)
display(scenario_summary_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(scenario_summary_df["scenario"], scenario_summary_df["n_series"], marker="o")
ax.set_title("Number of constructed series by scenario")
ax.set_ylabel("Number of series")
ax.set_xlabel("Scenario")
plt.show()

## 15. Save outputs

Files saved:

- `series_daily_wide.csv`: daily signed amount, one column per series;
- `series_daily_long.parquet`: daily long format;
- `series_info.csv`: metadata per series;
- `company_to_series.csv`: company-to-series audit mapping;
- `assigned_transactions.parquet`: original rows with assigned `series_id`;
- `company_features.csv`: company features and own-series decision.

In [ ]:
def save_outputs(outputs, company_features, cfg: SeriesConfig):
    outdir = Path(cfg.output_dir)
    outdir.mkdir(parents=True, exist_ok=True)

    outputs["series_wide"].to_csv(outdir / "series_daily_wide.csv")
    outputs["series_long"].to_parquet(outdir / "series_daily_long.parquet", index=False)
    outputs["series_info"].to_csv(outdir / "series_info.csv", index=False)
    outputs["company_to_series"].to_csv(outdir / "company_to_series.csv", index=False)
    outputs["assigned_transactions"].to_parquet(outdir / "assigned_transactions.parquet", index=False)
    company_features.to_csv(outdir / "company_features.csv", index=False)

    print(f"Saved outputs to: {outdir.resolve()}")


save_outputs(outputs, company_features, cfg)

## 16. How to use these series for forecasting later

Recommended next step:

1. Start with `series_daily_wide.csv`.
2. Forecast each selected series D+1.
3. Sum forecasts to the total.
4. Compare against:
   - direct aggregate forecast;
   - own-company-only + tail forecast;
   - compact / balanced / detailed scenarios.
5. Keep the granularity that improves the final D+1 treasury metric, not the one that merely looks more detailed.

For intermittent or rare-spike groups, use two-part logic:

```text
Expected amount tomorrow =
P(non-zero tomorrow) × E(amount tomorrow | non-zero)
```

For final production, you may want a reconciled hierarchy:

```text
Total
= under_100k + over_100k
= own companies + tail groups + NO_COMPANY
```

But before reconciliation, this notebook gives you the auditable construction layer.